# FutureWork AI — Expand Australian occupation mapping (review assistant)

**Purpose:** Generate up to three title-based ANZSCO candidates for **every eligible O*NET occupation**, show confidence indicators and export a review queue. Title similarity is a *screening aid*, not proof of occupational equivalence. Australian growth rates are the published **4-digit group projections**, not forecasts for each 6-digit occupation. The 2025 shortage list is a point-in-time measure, not a guarantee of openings.

**Required files:** `processed/occupation_lookup.csv`, `data/au_occupation_titles.csv`, `data/au_market_signals.csv`, optional `data/verified_mapping.csv`. The previously verified four prototype mappings are preserved when present.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, re, difflib
from IPython.display import display
ROOT=Path.cwd(); DATA=ROOT/'data'; PROCESSED=ROOT/'processed'; RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
lookup=pd.read_csv(PROCESSED/'occupation_lookup.csv',dtype={'occupation_code':str}).drop_duplicates('occupation_code')
au=pd.read_csv(DATA/'au_occupation_titles.csv',dtype={'au_code':str})
market=pd.read_csv(DATA/'au_market_signals.csv',dtype={'au_code':str})
assert {'occupation_code','occupation_title'}.issubset(lookup.columns)
assert {'au_code','au_title'}.issubset(au.columns)
assert au.au_code.str.fullmatch(r'\d{6}').all()
print(f'O*NET occupations: {len(lookup)}, Australian ANZSCO occupation titles: {len(au)}')

In [ ]:
# Match titles without converting guesses into approved mappings.
# token-based normalization helps with singular/plural/capitalization/ordering,
# while fuzzy string similarity proposes further candidates for human review.
def norm(t):
    t=re.sub(r'[^a-z0-9 ]',' ',str(t).lower())
    tokens=[x for x in t.split() if x not in {'and','or','the','of','other'}]
    return ' '.join(sorted(x[:-1] if x.endswith('s') and len(x)>4 else x for x in tokens))

try:
    from rapidfuzz import process, fuzz
    HAS_RAPID=True
except ImportError:
    HAS_RAPID=False

au=au.drop_duplicates('au_code').copy()
au['norm_title']=au.au_title.map(norm)
lookup['norm_title']=lookup.occupation_title.map(norm)
choices=au.norm_title.tolist()
rows=[]
for r in lookup.itertuples():
    if HAS_RAPID:
        cand=process.extract(r.norm_title,choices,scorer=fuzz.ratio,limit=3)
        hits=[(i, float(score)/100) for _,score,i in cand]
    else:
        sims=[difflib.SequenceMatcher(None,r.norm_title,a).ratio() for a in choices]
        hits=[(i,float(sims[i])) for i in np.argsort(sims)[-3:][::-1]]
    for rank,(i,score) in enumerate(hits,1):
        a=au.iloc[i]
        rows.append({'occupation_code':r.occupation_code,'occupation_title':r.occupation_title,
                     'candidate_rank':rank,'au_code':a.au_code,'au_title':a.au_title,
                     'title_similarity':round(score,3),
                     'match_status':'CANDIDATE_NOT_VERIFIED',
                     'growth_pct_10yr':a.growth_pct_10yr,
                     'shortage_status':a.shortage_status})
candidates=pd.DataFrame(rows)
candidates.to_csv(RESULTS/'all_occupation_mapping_candidates.csv',index=False)
print('Proposed candidate rows:',len(candidates),'unique source occupations:',candidates.occupation_code.nunique())
print('EXACT normalized title candidate count:',((candidates.candidate_rank==1)&(candidates.title_similarity==1)).sum())
display(candidates.head(12))

## Review queue, not an automatic approval

Check job **duties**, not just the title. Choose an ANZSCO code only when its definition reasonably represents the O*NET occupation. One-to-many or ambiguous matches should remain unreviewed. Data without verified mappings remains unknown in the live demand-aware ranking.

In [ ]:
# Keep only high-priority first candidates for review; every source occupation still has candidates in the complete CSV.
queue=candidates[candidates.candidate_rank==1].copy()
queue['needs_manual_review']=True
queue=queue.sort_values(['title_similarity','occupation_title'],ascending=[False,True])
queue.to_csv(RESULTS/'mapping_review_queue.csv',index=False)
display(queue[['occupation_title','au_title','title_similarity','match_status']].head(25))

In [ ]:
# OPTIONAL human-reviewed approvals, entered only after checking occupational duties.
# Example: approvals={'15-1254.00':'261212'}
approvals={}
reviewed_rows=[]
for onet,au_code in approvals.items():
    assert onet in set(lookup.occupation_code),f'Unknown O*NET code: {onet}'
    assert au_code in set(au.au_code),f'Unknown ANZSCO code: {au_code}'
    reviewed_rows.append({'occupation_code':onet,'au_code':au_code,'au_classification':'ANZSCO',
                          'reviewed':'yes','source_note':'Human-reviewed occupational duties; confirm and add source URLs'})
new=pd.DataFrame(reviewed_rows,columns=['occupation_code','au_code','au_classification','reviewed','source_note'])
oldpath=DATA/'verified_mapping.csv'
if oldpath.exists():
    old=pd.read_csv(oldpath,dtype=str).fillna('')
    old=old[old.reviewed.str.strip().str.lower().eq('yes')]
    updated=pd.concat([old,new],ignore_index=True).drop_duplicates('occupation_code',keep='last')
else:
    updated=new
updated.to_csv(RESULTS/'verified_mapping_updated.csv',index=False)
print('Approved mappings available:',len(updated))
print('Coverage of O*NET occupations:',f'{len(updated)/len(lookup):.1%}')
print('Copy results/verified_mapping_updated.csv to data/verified_mapping.csv ONLY after review.')
display(updated.head(10))

## Next step

1. Download `results/all_occupation_mapping_candidates.csv` and inspect candidates. For each questionable match, read both occupation definitions.
2. Approve only defensible equivalences in the `approvals` dictionary and rerun the last cell.
3. Replace `data/verified_mapping.csv` with the resulting `results/verified_mapping_updated.csv` **after** reviewing changes.
4. Restart and rerun `FutureWork_AI_Demand_Aware.ipynb`.

Do not label all 910 jobs as demand-verified unless all 910 have been individually mapped with defensible evidence. Keep unmatched entries as **unknown**. Be aware that the O*NET and Australian occupation systems are not necessarily one-to-one.